******Car-Dheko_Used_Car_Price_Prediction******

There are 6 files of cities data

****Data Processing****

**a)	Import and concatenate:**

i)	Import all city’s dataset which is in unstructured format.

ii)	Convert it into a  structured format.

iii)Added a new column named ‘City’ and assign values for all rows with the name of the respective city.

iv)	Concatenate all datasets and make it as a single dataset.

**Banglore**

Perfectly Done unstructured to structured Banglore cars

In [10]:
import pandas as pd 
import ast
import os

def load_and_parse_data(file_path):
    """Load and parse the CSV file with nested JSON data"""
    try:
        df = pd.read_csv(file_path)
        
        def parse_dict_column(column):
            return column.apply(lambda x: ast.literal_eval(x) if isinstance(x, str) and x.strip().startswith('{') else x)
        
        for col in ["new_car_detail", "new_car_overview", "new_car_feature", "new_car_specs"]:
            if col in df.columns:
                df[col] = parse_dict_column(df[col])
        
        return df
    except Exception as e:
        print(f"❌ Error loading file: {e}")
        return None

def flatten_data(df):
    """Flatten all nested structures into a single DataFrame"""
    try:
        # OVERVIEW
        def flatten_overview(overview_data):
            if isinstance(overview_data, dict):
                return {item['key']: item['value'] for item in overview_data.get("top", []) if isinstance(item, dict)}
            return {}
        
        df_overview = df["new_car_overview"].apply(flatten_overview) if "new_car_overview" in df.columns else pd.DataFrame()

        # DETAIL
        df_detail = pd.json_normalize(df["new_car_detail"]) if "new_car_detail" in df.columns else pd.DataFrame()

        # FEATURE
        def flatten_feature(data):
            features = []
            if isinstance(data, dict):
                top = data.get("top", [])
                features.extend(f.get("value") for f in top if isinstance(f, dict))
                for section in data.get("data", []):
                    if isinstance(section, dict):
                        for item in section.get("list", []):
                            if isinstance(item, dict):
                                val = item.get("value")
                                if val:
                                    features.append(val)
            return {'Feature_' + str(i): v for i, v in enumerate(features)}
        
        df_feature = df["new_car_feature"].apply(flatten_feature) if "new_car_feature" in df.columns else pd.DataFrame()

        # SPECS
        def flatten_specs(data):
            specs = {}
            if isinstance(data, dict):
                for item in data.get("top", []):
                    if isinstance(item, dict):
                        k = item.get("key")
                        v = item.get("value")
                        if k and v:
                            specs[k] = v
                for section in data.get("data", []):
                    if isinstance(section, dict):
                        for item in section.get("list", []):
                            if isinstance(item, dict):
                                k = item.get("key")
                                v = item.get("value")
                                if k and v:
                                    specs[k] = v
            return specs
        
        df_specs = df["new_car_specs"].apply(flatten_specs) if "new_car_specs" in df.columns else pd.DataFrame()

        # Convert all dict results to DataFrames
        df_overview = pd.DataFrame(df_overview.tolist())
        df_feature = pd.DataFrame(df_feature.tolist())
        df_specs = pd.DataFrame(df_specs.tolist())

        # Combine all DataFrames
        base_cols = df[["car_links"]] if "car_links" in df.columns else pd.DataFrame()
        df_final = pd.concat([base_cols, df_detail, df_overview, df_feature, df_specs], axis=1)
        df_final["City"] = "bangalore"
        
        return df_final
    except Exception as e:
        print(f"❌ Error flattening data: {e}")
        return None

def save_cleaned_data(df, output_dir="output"):
    """Save the cleaned DataFrame to csv"""
    try:
        os.makedirs(output_dir, exist_ok=True)
        output_path = os.path.join(output_dir, "structured_bangalore_cars.csv")
        df.to_csv(output_path, index=False)
        print(f"✅ Saved cleaned data to: {output_path}")
        return True
    except Exception as e:
        print(f"❌ Error saving data: {e}")
        return False

def main():
    input_path = "csv_files/bangalore_cars.csv"
    df = load_and_parse_data(input_path)
    if df is None:
        return

    df_flat = flatten_data(df)
    if df_flat is None:
        return

    save_cleaned_data(df_flat)

if __name__ == "__main__":
    main()


✅ Saved cleaned data to: output\structured_bangalore_cars.csv


In [14]:
df_final["City"] 

0       bangalore
1       bangalore
2       bangalore
3       bangalore
4       bangalore
          ...    
1476    bangalore
1477    bangalore
1478    bangalore
1479    bangalore
1480    bangalore
Name: City, Length: 1481, dtype: object

             Column Data Type  Non-Null Count  NaN Count  NaN Percentage
0    new_car_detail    object            1481          0             0.0
1  new_car_overview    object            1481          0             0.0
2   new_car_feature    object            1481          0             0.0
3     new_car_specs    object            1481          0             0.0
4         car_links    object            1481          0             0.0


In [15]:
nan_summary = pd.DataFrame({
    'Column': df_final .columns,
    'Data Type': df_final .dtypes,
    'Non-Null Count': df_final.notna().sum(),
    'NaN Count': df_final .isna().sum(),
    'NaN Percentage': (df_final.isna().mean() * 100).round(2)
}).reset_index(drop=True)

# Display ALL rows without truncation
with pd.option_context('display.max_rows', None, 'display.width', 1000):
    print(nan_summary)

                       Column Data Type  Non-Null Count  NaN Count  NaN Percentage
0                          it     int64            1481          0            0.00
1                          ft    object            1481          0            0.00
2                          bt    object            1481          0            0.00
3                          km    object            1481          0            0.00
4                transmission    object            1481          0            0.00
5                     ownerNo     int64            1481          0            0.00
6                       owner    object            1481          0            0.00
7                         oem    object            1481          0            0.00
8                       model    object            1481          0            0.00
9                   modelYear     int64            1481          0            0.00
10           centralVariantId     int64            1481          0            0.00
11  

In [34]:
df_final[['price','bt', 'Kms Driven','owner' ,'Color','Year of Manufacture','Mileage', 'RTO','Fuel Type','Registration Year','modelYear', 'Insurance Validity','Gear Box', 'modelYear', 'Transmission', 'Seats', 'City', 'Engine Displacement']].head(10)

,price,bt,Kms Driven,owner,Color,Year of Manufacture,Mileage,RTO,Fuel Type,Registration Year,modelYear,Insurance Validity,Gear Box,modelYear,Transmission,Seats,Seats,City,Engine Displacement
0,₹ 4 Lakh,Hatchback,"1,20,000 Kms",3rd Owner,White,2015.0,23.1 kmpl,KA51,Petrol,2015,2015,Third Party insurance,5 Speed,2015,Manual,5 Seats,5,bangalore,998 cc
1,₹ 8.11 Lakh,SUV,"32,706 Kms",2nd Owner,White,2018.0,17 kmpl,KA05,Petrol,Feb 2018,2018,Comprehensive,5 Speed,2018,Manual,5 Seats,5,bangalore,1497 cc
2,₹ 5.85 Lakh,Hatchback,"11,949 Kms",1st Owner,Red,2018.0,23.84 kmpl,KA03,Petrol,Sept 2018,2018,Comprehensive,5 Speed,2018,Manual,5 Seats,5,bangalore,1199 cc
3,₹ 4.62 Lakh,Sedan,"17,794 Kms",1st Owner,Others,2014.0,19.1 kmpl,KA53,Petrol,Dec 2014,2014,Comprehensive,5 Speed,2014,Manual,5 Seats,5,bangalore,1197 cc
4,₹ 7.90 Lakh,SUV,"60,000 Kms",1st Owner,Gray,2015.0,23.65 kmpl,KA04,Diesel,2015,2015,Third Party insurance,5 Speed,2015,Manual,5 Seats,5,bangalore,1248 cc
5,₹ 19 Lakh,SUV,"20,000 Kms",1st Owner,Others,2020.0,17.1 kmpl,KA04,Diesel,2020,2020,Third Party insurance,6 Speed,2020,Manual,5 Seats,5,bangalore,1956 cc
6,₹ 3.45 Lakh,Hatchback,"37,772 Kms",1st Owner,Grey,2017.0,20.63 kmpl,KA05,Petrol,Aug 2017,2017,Comprehensive,5 Speed,2017,Manual,5 Seats,5,bangalore,1198 cc
7,₹ 12 Lakh,SUV,"30,000 Kms",1st Owner,Others,2021.0,18.15 kmpl,KA51,Petrol,2021,2021,Third Party insurance,7-Speed,2021,Automatic,5 Seats,5,bangalore,998 cc
8,₹ 9.60 Lakh,Sedan,"37,000 Kms",1st Owner,Maroon,2018.0,20.28 kmpl,KA03,Petrol,Aug 2018,2018,Comprehensive,4 Speed,2018,Automatic,5 Seats,5,bangalore,1462 cc
9,₹ 5.85 Lakh,Hatchback,"11,949 Kms",1st Owner,Red,2017.0,23.84 kmpl,KA03,Petrol,Jan 2018,2017,Comprehensive,5 Speed,2017,Manual,5 Seats,5,bangalore,1199 cc


✅ Selected Features and Justifications

| Column               | Description                                   | Justification                                                                 |
|----------------------|-----------------------------------------------|-------------------------------------------------------------------------------|
| `price`              | Selling price of the used car (Target variable) | This is the variable we're predicting.                                        |
| `bt`                 | Possibly body type or build type              | Vehicle type (e.g., SUV, sedan) affects demand, pricing, and buyer preference.|
| `Kms Driven`         | Total kilometers driven                        | Indicates vehicle usage; higher values usually reduce resale value.           |
| `owner`              | Number or type of previous owners              | Helps assess usage history and trust; fewer owners often mean better value.   |
| `Color`              | Exterior color of the car                      | Certain colors have higher resale appeal depending on regional trends.        |
| `Year of Manufacture`| Production year of the car                     | Indicates age; newer cars typically sell for higher prices.                   |
| `Mileage`            | Fuel efficiency (e.g., km/l)                   | Higher mileage is appealing and adds value to the car.                        |
| `RTO`                | Regional Transport Office location             | RTO location can influence resale value due to local tax rates and rules.     |
| `Fuel Type`          | Petrol, Diesel, CNG, Electric, etc.            | Different fuels affect running costs and buyer demand.                        |
| `Registration Year`  | Year the car was registered                    | Might differ from manufacturing year; important for insurance and resale.     |
| `modelYear`          | Year of manufacture                            | Reflects the age of the vehicle; newer cars tend to sell for higher prices.   |
| `Insurance Validity` | Remaining insurance period                     | A valid insurance policy adds value and trust for the buyer.                  |
| `Gear Box`           | Number of gears                                | Indicates car performance and class; more gears can mean better performance.  |
| `Transmission`       | Manual or Automatic                            | Automatics generally command a higher resale price, especially in cities.     |
| `Seats`              | Number of seats                                | More seating capacity appeals to families and commercial buyers.              |
| `City`               | City where the car is listed                   | Price trends vary by location due to demand, taxes, and road conditions.      |
| `Engine Displacement`| Engine size in CC                              | Affects performance, tax class, and buyer interest.                           |


Banglore necessary  columns

In [15]:
import pandas as pd
import os

# Load the Excel file
file_path = "output/structured_bangalore_cars.csv"
df = pd.read_csv(file_path)

# Define wanted columns
wanted_columns = [

    'price',
    'bt',
    'Kms Driven',
    'owner' ,
    
    'Year of Manufacture',
    'Mileage',
    'RTO',
    'Fuel Type',
    'Registration Year',
    'modelYear',
    'Insurance Validity',
    'Gear Box',
    'Transmission',
    'Seats',
    'City',
    'Engine Displacement'
  
]

# Filter only the columns that exist in the DataFrame
wanted_columns_present = [col for col in wanted_columns if col in df.columns]

# Select and save the cleaned data
df_cleaned = df[wanted_columns_present]

# Make sure the output folder exists
os.makedirs("output", exist_ok=True)

# Save the DataFrame to Excel inside the output folder
df_cleaned.to_csv("output/banglore_wanted.csv", index=False)

print("✅ File saved to: output/banglore_wanted.csv")


✅ File saved to: output/banglore_wanted.csv


In [16]:
nan_summary = pd.DataFrame({
    'Column': df_cleaned .columns,
    'Data Type': df_cleaned .dtypes,
    'Non-Null Count': df_cleaned.notna().sum(),
    'NaN Count': df_cleaned .isna().sum(),
    'NaN Percentage': (df_cleaned.isna().mean() * 100).round(2)
}).reset_index(drop=True)

# Display ALL rows without truncation
with pd.option_context('display.max_rows', None, 'display.width', 1000):
    print(nan_summary)

                 Column Data Type  Non-Null Count  NaN Count  NaN Percentage
0                 price    object            1481          0            0.00
1                    bt    object            1481          0            0.00
2            Kms Driven    object            1481          0            0.00
3                 owner    object            1481          0            0.00
4   Year of Manufacture   float64            1474          7            0.47
5               Mileage    object            1439         42            2.84
6                   RTO    object            1313        168           11.34
7             Fuel Type    object            1481          0            0.00
8     Registration Year    object            1474          7            0.47
9             modelYear     int64            1481          0            0.00
10   Insurance Validity    object            1478          3            0.20
11             Gear Box    object            1457         24            1.62

***b)Handling Missing Values:***

 Identify and fill or remove missing values in the dataset. 

i)	For numerical columns, use techniques like mean, median, or mode imputation.

ii)	For categorical columns, use mode imputation or create a new category for missing values.


In [17]:
import pandas as pd
import numpy as np
import os

# Load the dataset
df = pd.read_csv("output/banglore_wanted.csv")

# Function to print missing value statistics
def print_missing_stats(df, title="Before Imputation"):
    print(f"\n🔍 {title} Missing Values Summary:")
    print("="*60)
    missing_data = df.isnull().sum()
    total_rows = len(df)
    missing_percent = (missing_data / total_rows) * 100
    
    stats_df = pd.DataFrame({
        'Missing Values': missing_data,
        '% Missing': missing_percent.round(2)
    })
    
    print(stats_df[stats_df['Missing Values'] > 0].sort_values('% Missing', ascending=False))
    print("="*60)
    print(f"Total rows in dataset: {total_rows}\n")

# Initial missing value analysis
print_missing_stats(df)

# Create a copy for tracking changes
df_cleaned_filled = df.copy()

# Dictionary to store imputation details
imputation_report = {}

# Handle numerical columns
numerical_cols = ['Year of Manufacture']
for col in numerical_cols:
    if col in df_cleaned_filled.columns:
        before = df_cleaned_filled[col].isnull().sum()
        median_val = df_cleaned_filled[col].median()
        df_cleaned_filled[col].fillna(median_val, inplace=True)
        after = df_cleaned_filled[col].isnull().sum()
        
        imputation_report[col] = {
            'type': 'numerical',
            'before': before,
            'filled': before - after,
            'method': 'median',
            'value': median_val,
            'justification': 'Median is robust against outliers in manufacturing years'
        }

# Handle categorical columns
categorical_cols = ['Color', 'RTO', 'Registration Year', 'Insurance Validity', 
                   'Gear Box', 'Seats', 'Engine Displacement', 'Mileage']

for col in categorical_cols:
    if col in df_cleaned_filled.columns:
        before = df_cleaned_filled[col].isnull().sum()
        mode_val = df_cleaned_filled[col].mode()[0]
        df_cleaned_filled[col].fillna(mode_val, inplace=True)
        after = df_cleaned_filled[col].isnull().sum()
        
        imputation_report[col] = {
            'type': 'categorical',
            'before': before,
            'filled': before - after,
            'method': 'mode',
            'value': mode_val,
            'justification': 'Most frequent value is appropriate for categorical data'
        }

# Special handling for 'Mileage' column
if 'Mileage' in df_cleaned_filled.columns:
    # Extract numerical part from Mileage (e.g., '23.1 kmpl' → 23.1)
    df_cleaned_filled['Mileage'] = df_cleaned_filled['Mileage'].str.extract('(\d+\.?\d*)').astype(float)
    
    before = df_cleaned_filled['Mileage'].isnull().sum()
    median_mileage = df_cleaned_filled['Mileage'].median()
    df_cleaned_filled['Mileage'].fillna(median_mileage, inplace=True)
    after = df_cleaned_filled['Mileage'].isnull().sum()
    
    imputation_report['Mileage'] = {
        'type': 'converted numerical',
        'before': before,
        'filled': before - after,
        'method': 'median',
        'value': median_mileage,
        'justification': 'After converting string to numerical, median is robust for mileage values'
    }

# Final missing value analysis
print_missing_stats(df_cleaned_filled, "After Imputation")

# Generate imputation report
print("\n✅ Imputation Summary")
print("="*60)
print("Here's a breakdown of how missing values were handled in your dataset:")
print("-"*60)

report_df = pd.DataFrame.from_dict(imputation_report, orient='index')
report_df = report_df[['type', 'before', 'filled', 'method', 'value', 'justification']]
report_df.columns = ['Type', 'Missing Before', 'Filled', 'Strategy', 'Value Used', 'Justification']

print(report_df.sort_values('Missing Before', ascending=False))
print("="*60)

# Detect and print data types
print("\n🔎 Final Data Types:")
print("="*60)
print(df_cleaned_filled.dtypes)
print("="*60)

# df_cleaned_filled = df_cleaned_filled.convert_dtypes()  # Automatically converts object → string, numbers → Int64/Float64
# print(df_cleaned_filled.dtypes)
# print("automatically_detected the datatypes")

# Create a copy to detect data types without modifying the original
df_detected_dtypes = df_cleaned_filled.copy().convert_dtypes()

print("Original DataFrame dtypes:")
print(df_cleaned_filled.dtypes)
print("\nDetected dtypes (without saving):")
print(df_detected_dtypes.dtypes)
print("\nAutomatically detected the datatypes (in the copy)")

# Save the cleaned data
os.makedirs("output", exist_ok=True)
output_path = "output/banglore_filled_removed.csv"
df_detected_dtypes.to_csv(output_path, index=False)

print(f"\n✅ Cleaned dataset saved to: {output_path}")


🔍 Before Imputation Missing Values Summary:
                     Missing Values  % Missing
RTO                             168      11.34
Mileage                          42       2.84
Gear Box                         24       1.62
Year of Manufacture               7       0.47
Registration Year                 7       0.47
Insurance Validity                3       0.20
Engine Displacement               3       0.20
Seats                             1       0.07
Total rows in dataset: 1481


🔍 After Imputation Missing Values Summary:
Empty DataFrame
Columns: [Missing Values, % Missing]
Index: []
Total rows in dataset: 1481


✅ Imputation Summary
Here's a breakdown of how missing values were handled in your dataset:
------------------------------------------------------------
                                    Type  Missing Before  Filled Strategy  \
RTO                          categorical             168     168     mode   
Gear Box                     categorical              24   

C:\Users\USER\AppData\Local\Temp\ipykernel_7492\255457326.py:40: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_cleaned_filled[col].fillna(median_val, inplace=True)
C:\Users\USER\AppData\Local\Temp\ipykernel_7492\255457326.py:60: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For 

In [18]:
nan_summary = pd.DataFrame({
    'Column': df_detected_dtypes .columns,
    'Data Type': df_detected_dtypes .dtypes,
    'Non-Null Count': df_detected_dtypes.notna().sum(),
    'NaN Count': df_detected_dtypes .isna().sum(),
    'NaN Percentage': (df_detected_dtypes.isna().mean() * 100).round(2)
}).reset_index(drop=True)

# Display ALL rows without truncation
with pd.option_context('display.max_rows', None, 'display.width', 1000):
    print(nan_summary)

                 Column       Data Type  Non-Null Count  NaN Count  NaN Percentage
0                 price  string[python]            1481          0             0.0
1                    bt  string[python]            1481          0             0.0
2            Kms Driven  string[python]            1481          0             0.0
3                 owner  string[python]            1481          0             0.0
4   Year of Manufacture           Int64            1481          0             0.0
5               Mileage         Float64            1481          0             0.0
6                   RTO  string[python]            1481          0             0.0
7             Fuel Type  string[python]            1481          0             0.0
8     Registration Year  string[python]            1481          0             0.0
9             modelYear           Int64            1481          0             0.0
10   Insurance Validity  string[python]            1481          0             0.0
11  

***c)	Standardising Data Formats:***

i)	Check for all data types and do the necessary steps to keep the data in the correct format.

(1)	Eg. If a data point has string formats like 70 kms, then remove the unit ‘kms’ and change the data type from string to integers.


In [19]:
df_detected_dtypes.head()

,price,bt,Kms Driven,owner,Year of Manufacture,Mileage,RTO,Fuel Type,Registration Year,modelYear,Insurance Validity,Gear Box,Transmission,Seats,City,Engine Displacement
0,₹ 4 Lakh,Hatchback,"1,20,000 Kms",3rd Owner,2015,23.1,KA51,Petrol,2015,2015,Third Party insurance,5 Speed,Manual,5 Seats,bangalore,998 cc
1,₹ 8.11 Lakh,SUV,"32,706 Kms",2nd Owner,2018,17.0,KA05,Petrol,Feb 2018,2018,Comprehensive,5 Speed,Manual,5 Seats,bangalore,1497 cc
2,₹ 5.85 Lakh,Hatchback,"11,949 Kms",1st Owner,2018,23.84,KA03,Petrol,Sept 2018,2018,Comprehensive,5 Speed,Manual,5 Seats,bangalore,1199 cc
3,₹ 4.62 Lakh,Sedan,"17,794 Kms",1st Owner,2014,19.1,KA53,Petrol,Dec 2014,2014,Comprehensive,5 Speed,Manual,5 Seats,bangalore,1197 cc
4,₹ 7.90 Lakh,SUV,"60,000 Kms",1st Owner,2015,23.65,KA04,Diesel,2015,2015,Third Party insurance,5 Speed,Manual,5 Seats,bangalore,1248 cc


In [39]:
import pandas as pd
from decimal import Decimal
import os
import re


try:
    df_standardised = pd.read_csv("output/banglore_filled_removed.csv")  # Ensure correct file name
except FileNotFoundError:
    print("Error: The file 'output/banglore_filled_removed.csv' was not found.")
    exit()

print(" the datatypes before transformation")
print(df_standardised.dtypes)

def clean_numeric_column(series, pattern=None, dtype=float):
    """Helper function to clean numeric columns"""
    try:
        if series.dtype == object:
            if pattern:
                extracted = series.str.extract(pattern)[0]
            else:
                extracted = series.astype(str).str.replace(r'[^\d\.]', '', regex=True) # Keep decimals
            if dtype == 'Int64':
                return pd.to_numeric(extracted, errors='coerce').astype('Int64')
            return pd.to_numeric(extracted, errors='coerce')
        return series
    except Exception as e:
        print(f"⚠️ Warning cleaning column {series.name}: {e}")
        return series
    
def clean_numeric_series(series, dtype=float, pattern=None):
    """Clean a pandas Series containing numeric values"""
    try:
        if not pd.api.types.is_string_dtype(series):
            series = series.astype(str)
            
        # Remove commas and extract numeric values
        cleaned = series.str.replace(',', '')
        
        if pattern:
            extracted = cleaned.str.extract(pattern)[0]
        else:
            if dtype == float:
                extracted = cleaned.str.extract(r'([\d\.]+)')[0]
            else:
                extracted = cleaned.str.extract(r'(\d+)')[0]
        
        # Convert to appropriate type
        if dtype == 'Int64':
            return pd.to_numeric(extracted, errors='coerce').astype('Int64')
        return pd.to_numeric(extracted, errors='coerce').astype(dtype)
    except Exception as e:
        print(f"⚠️ Warning cleaning numeric series: {e}")
        return series
    
def clean_kms_driven(kms_str):
    """
    Cleans the 'Kms Driven' string by removing non-numeric characters
    and converting it to an integer.
    Handles cases with commas and the "Kms" suffix.
    Returns the cleaned number as an integer or None if cleaning fails.
    """
    if isinstance(kms_str, (int, float)):
        return int(kms_str)  # Already numeric

    cleaned_str = re.sub(r'[^\d]', '', str(kms_str))  # Remove non-digits
    if cleaned_str:
        return int(cleaned_str)
    return None

if 'Kms Driven' in df_standardised.columns:
    df_standardised['Kms Driven'] = df_standardised['Kms Driven'].apply(clean_kms_driven)
    print("✅ Processed 'Kms Driven' column - removed non-numeric characters and converted to integer")



def convert_lakh_to_numeric_v4(price_str):
    """Converts price strings in '₹ X.XX Lakh' or '₹ X Lakh' format to numeric INR (more robust).
    Handles potential non-numeric values by returning the original string.
    """
    if isinstance(price_str, (int, float, Decimal)):
        return float(price_str)  # Already numeric

    price_str = str(price_str).strip()  # Remove leading/trailing whitespace

    match = re.search(r'₹\s*(\d+(\.\d+)?)\s*Lakh', price_str, re.IGNORECASE)
    if match:
        return float(Decimal(match.group(1)) * Decimal('100000'))
    else:
        return price_str  # Return the original string if no match

if 'price' in df_standardised.columns:
    original_prices = df_standardised['price'].copy()
    df_standardised['price_numeric'] = df_standardised['price'].apply(convert_lakh_to_numeric_v4)

    # Identify non-convertible values (still strings after the function)
    non_convertible = df_standardised[pd.to_numeric(df_standardised['price_numeric'], errors='coerce').isna()]['price'].unique().tolist()

    # Update the original 'price' column with the numeric conversions
    df_standardised['price'] = pd.to_numeric(df_standardised['price_numeric'], errors='coerce')

    print("✅ Processed 'price' column - attempted to convert 'Lakh' values to numeric INR (version 4)")
    if non_convertible:
        print(f"\n⚠️ The following original values in the 'price' column could not be converted to numeric: {non_convertible}")
    else:
        print("\n🎉 All 'Lakh' values in the 'price' column were successfully converted to numeric.")

    # Remove the temporary 'price_numeric' column
    df_standardised.drop(columns=['price_numeric'], inplace=True)

    print(f"\nNumber of NaN values in 'price' column after conversion: {df_standardised['price'].isnull().sum()}")


# def convert_lakh_to_numeric_v3(price_str):
#     """Converts price strings in '₹ X.XX Lakh' or '₹ X Lakh' format to numeric INR (more robust)."""
#     if isinstance(price_str, (int, float)):
#         return float(price_str)  # Already numeric

#     price_str = str(price_str).strip()  # Remove leading/trailing whitespace

#     match = re.search(r'₹\s*(\d+(\.\d+)?)\s*Lakh', price_str, re.IGNORECASE)
#     if match:
#         return float(Decimal(match.group(1)) * Decimal('100000'))
#     return None

# if 'price' in df_standardised.columns:
#     df_standardised['price'] = df_standardised['price'].apply(convert_lakh_to_numeric_v3)
#     print("✅ Processed 'price' column - converted 'Lakh' values to numeric INR (version 3)")



# def convert_lakh_to_numeric(price_str):
#     """Converts a price string in '₹ X.XX Lakh' format to a numeric value."""
#     if isinstance(price_str, (int, float)):
#         return price_str  # Already numeric

#     match = re.search(r'₹\s*(\d+(\.\d+)?)\s*Lakh', str(price_str), re.IGNORECASE)
#     if match:
#         return float(Decimal(match.group(1)) * Decimal('100000'))
#     return None  # Or handle cases where the format doesn't match differently

# if 'price' in df_standardised.columns:
#     df_standardised['price'] = df_standardised['price'].apply(convert_lakh_to_numeric)
#     print("✅ Processed 'price' - converted 'Lakh' values to numeric INR")

# def extract_gear_speeds(gear_box_str):
#     """
#     Extracts the first number representing the number of speeds from a Gear Box string.
#     Handles cases with "Speed", "-", and other characters.
#     Returns the extracted number as an integer or None if no number is found.
#     """
#     if isinstance(gear_box_str, (int, float)):
#         return int(gear_box_str)  # Already a number

#     match = re.search(r'(\d+)', str(gear_box_str))
#     if match:
#         return int(match.group(1))
#     return None

# if 'Gear Box' in df_standardised.columns:
#     df_standardised['Gear Box'] = df_standardised['Gear Box'].apply(extract_gear_speeds)
#     print("✅ Processed 'Gear Box' column - extracted the number of speeds")


# def extract_gear_speeds_v2(gear_box_str):
#     if isinstance(gear_box_str, (int, float)):
#         return int(gear_box_str)

#     match = re.search(r'(\d+)\s*Speed', str(gear_box_str), re.IGNORECASE)
#     if match:
#         return int(match.group(1))

#     match = re.search(r'(\d+)-Speed', str(gear_box_str), re.IGNORECASE)
#     if match:
#         return int(match.group(1))

#     match = re.search(r'^(\d+)', str(gear_box_str)) # Check for leading number if others fail
#     if match:
#         return int(match.group(1))

#     return None

# if 'Gear Box' in df_standardised.columns:
#     df_standardised['Gear Box'] = df_standardised['Gear Box'].apply(extract_gear_speeds_v2)
#     print("✅ Processed 'Gear Box' column - extracted the nr of speeds (version 2)")


def extract_gear_speeds_v3(gear_box_str):
    """
    Extracts the number of speeds from a Gear Box string.
    Handles cases with "Speed", "-", leading numbers, and returns original if no match.
    """
    if isinstance(gear_box_str, (int, float)):
        return int(gear_box_str)

    gear_box_str = str(gear_box_str).strip()

    patterns = [
        r'(\d+)\s*Speed',  # e.g., "6 Speed"
        r'(\d+)-Speed',   # e.g., "5-Speed"
        r'^(\d+)',         # Leading number, e.g., "8G-DCT" (captures 8)
    ]

    for pattern in patterns:
        match = re.search(pattern, gear_box_str, re.IGNORECASE)
        if match:
            return int(match.group(1))

    return gear_box_str  # Return original string if no pattern matches

if 'Gear Box' in df_standardised.columns:
    original_gear_box = df_standardised['Gear Box'].copy()
    df_standardised['Gear Box_numeric'] = df_standardised['Gear Box'].apply(extract_gear_speeds_v3)

    # Identify non-convertible values (still strings after the function)
    non_convertible_gear_box = df_standardised[pd.to_numeric(df_standardised['Gear Box_numeric'], errors='coerce').isna()]['Gear Box'].unique().tolist()

    # Update the original 'Gear Box' column with the numeric conversions
    df_standardised['Gear Box'] = pd.to_numeric(df_standardised['Gear Box_numeric'], errors='coerce')

    print("✅ Processed 'Gear Box' column - attempted to extract the number of speeds (version 3)")
    if non_convertible_gear_box:
        print(f"\n⚠️ The following original values in the 'Gear Box' column could not be converted to numeric: {non_convertible_gear_box}")
    else:
        print("\n🎉 All identifiable speed numbers in the 'Gear Box' column were successfully extracted.")

    print(f"\nNumber of NaN values in 'Gear Box' column after extraction: {df_standardised['Gear Box'].isnull().sum()}")

    # Remove the temporary 'Gear Box_numeric' column
    df_standardised.drop(columns=['Gear Box_numeric'], inplace=True)

def clean_and_transform(df_standardised):
    """Perform all cleaning and transformation operations"""
    try:
        df_transformed = df_standardised.copy()

        # ===== PROCESS KM COLUMN =====
        if 'Kms Driven' in df_standardised.columns:
            df_standardised['Kms Driven'] = clean_numeric_column(df_standardised['Kms Driven'], 'Int64')
            print("✅ Processed 'Kms Driven' column - removed commas and converted to integer")

        # ===== STANDARD TRANSFORMATIONS =====
        transformations = [
            #('price', r'â‚¹\s*(\d+(\.\d+)?)\s*Lakh', lambda x: float(Decimal(x.group(1)) * Decimal('100000'))),
            #('price', r'(\d+)', lambda x: float(Decimal(x.group(1)) * Decimal('100000'))),
            
            ('owner', r'(\d+)', 'Int64'),
            #('owner', None, lambda x: clean_numeric_column(x, dtype='Int64')), # Incorrect - owner is categorical
            #('Registration Year', None, lambda x: pd.to_datetime(x, errors='coerce').dt.year.astype('Int64')),
            ('Registration Year', r'(\d+)', 'Int64'),
            #('Gear Box', None, lambda x: clean_numeric_column(x, dtype='Int64')), # Incorrect - Gear Box is categorical
            
            ('Seats', r'(\d+)', 'Int64'),
            ('Engine Displacement', r'(\d+)', 'Int64'),
            ('Mileage', r'(\d+\.?\d*)', float) # Extract numerical mileage
        ]

    

        for col, pattern, action in transformations:
            if col in df_transformed.columns:
                try:
                    if callable(action):
                        if pattern:
                            match = df_transformed[col].str.extract(pattern)
                            df_transformed[col] = match[0].apply(action) if not match.empty else None
                        else:
                            df_transformed[col] = df_transformed[col].apply(action)
                    else:
                        df_transformed[col] = clean_numeric_column(df_transformed[col], pattern, action)
                except Exception as e:
                    print(f"⚠️ Warning processing column {col}: {e}")

        return df_transformed
    except Exception as e:
        print(f"❌ Error in transformation: {e}")
        return None




df_standardised = clean_and_transform(df_standardised)

# Assuming df_standardised is your DataFrame

year_columns_to_convert = ['Year of Manufacture', 'Registration Year', 'modelYear']

for col in year_columns_to_convert:
    if col in df_standardised.columns:
        try:
            df_standardised[col] = pd.to_datetime(df_standardised[col], format='%Y').dt.to_period('Y')
            print(f"Converted '{col}' to Period[Y]")
        except ValueError:
            print(f"Could not convert '{col}' to Period[Y], keeping original type.")

print("\nDataFrame dtypes after converting year columns:")
print(df_standardised.dtypes)

print("\nDataFrame with Period[Y] dtype:")
print(df_standardised[year_columns_to_convert].head())



# Create a copy to detect data types without modifying the original
df_standardised_dtypes = df_standardised.copy().convert_dtypes()

print("Original DataFrame dtypes:")
print(df_standardised.dtypes)
print("\nDetected dtypes (without saving):")
print(df_standardised_dtypes.dtypes)
print("\nAutomatically detected the datatypes (in the copy)")



if df_standardised_dtypes is not None:
    print("\n the datatypes after transformation")
    print(df_standardised_dtypes.dtypes)
    df_standardised_dtypes.head()

    # Save the df_Standardising data
    os.makedirs("output", exist_ok=True)
    output_path = "output/banglore_standardised.csv"
    df_standardised_dtypes.to_csv(output_path, index=False)
    print(f"\n✅ Cleaned dataset saved to: {output_path}")


 the datatypes before transformation
price                   object
bt                      object
Kms Driven              object
owner                   object
Year of Manufacture      int64
Mileage                float64
RTO                     object
Fuel Type               object
Registration Year       object
modelYear                int64
Insurance Validity      object
Gear Box                object
Transmission            object
Seats                   object
City                    object
Engine Displacement     object
dtype: object
✅ Processed 'Kms Driven' column - removed non-numeric characters and converted to integer
✅ Processed 'price' column - attempted to convert 'Lakh' values to numeric INR (version 4)

⚠️ The following original values in the 'price' column could not be converted to numeric: ['₹ 58,000 ', '₹ 55,000 ', '₹ 70,000 ', '₹ 40,000 ', '₹ 80,000 ', '₹ 1.30 Crore', '₹ 75,000 ', '₹ 65,000 ', '₹ 56,000 ', '₹ 60,000 ', '₹ 76,000 ', '₹ 50,000 ']

Number of NaN values

In [40]:
df_standardised_dtypes.head(10)

,price,bt,Kms Driven,owner,Year of Manufacture,Mileage,RTO,Fuel Type,Registration Year,modelYear,Insurance Validity,Gear Box,Transmission,Seats,City,Engine Displacement
0,400000,Hatchback,120000,3,2015,23.1,KA51,Petrol,2015,2015,Third Party insurance,5,Manual,5,bangalore,998
1,811000,SUV,32706,2,2018,17.0,KA05,Petrol,2018,2018,Comprehensive,5,Manual,5,bangalore,1497
2,585000,Hatchback,11949,1,2018,23.84,KA03,Petrol,2018,2018,Comprehensive,5,Manual,5,bangalore,1199
3,462000,Sedan,17794,1,2014,19.1,KA53,Petrol,2014,2014,Comprehensive,5,Manual,5,bangalore,1197
4,790000,SUV,60000,1,2015,23.65,KA04,Diesel,2015,2015,Third Party insurance,5,Manual,5,bangalore,1248
5,1900000,SUV,20000,1,2020,17.1,KA04,Diesel,2020,2020,Third Party insurance,6,Manual,5,bangalore,1956
6,345000,Hatchback,37772,1,2017,20.63,KA05,Petrol,2017,2017,Comprehensive,5,Manual,5,bangalore,1198
7,1200000,SUV,30000,1,2021,18.15,KA51,Petrol,2021,2021,Third Party insurance,7,Automatic,5,bangalore,998
8,960000,Sedan,37000,1,2018,20.28,KA03,Petrol,2018,2018,Comprehensive,4,Automatic,5,bangalore,1462
9,585000,Hatchback,11949,1,2017,23.84,KA03,Petrol,2018,2017,Comprehensive,5,Manual,5,bangalore,1199


In [42]:
nan_summary = pd.DataFrame({
    'Column': df_standardised_dtypes .columns,
    'Data Type': df_standardised_dtypes .dtypes,
    'Non-Null Count': df_standardised_dtypes.notna().sum(),
    'NaN Count': df_standardised_dtypes .isna().sum(),
    'NaN Percentage': (df_standardised_dtypes.isna().mean() * 100).round(2)
}).reset_index(drop=True)

# Display ALL rows without truncation
with pd.option_context('display.max_rows', None, 'display.width', 1000):
    print(nan_summary)

                 Column       Data Type  Non-Null Count  NaN Count  NaN Percentage
0                 price           Int64            1463         18            1.22
1                    bt  string[python]            1481          0            0.00
2            Kms Driven           Int64            1481          0            0.00
3                 owner           Int64            1481          0            0.00
4   Year of Manufacture   period[Y-DEC]            1481          0            0.00
5               Mileage         Float64            1481          0            0.00
6                   RTO  string[python]            1481          0            0.00
7             Fuel Type  string[python]            1481          0            0.00
8     Registration Year   period[Y-DEC]            1481          0            0.00
9             modelYear   period[Y-DEC]            1481          0            0.00
10   Insurance Validity  string[python]            1481          0            0.00
11  